# 22 — Family 2 (F2): One-Stage Anchor-Based Detector (`RetinaNet R50-FPN v2`)

**Experiments under Fixed Protocol P0 (`k5_drop_L1920`, `1.5 GPU-h`, `max_size=1280`):**
- **`retina-e0` (Baseline):** COCO-pretrained `retinanet_resnet50_fpn_v2`, default anchors (`32..812 px`), `detections_per_img=500`, `score_thresh=0.001`.
- **`retina-e1` (Exp A — EDA-Grounded K-Means Anchors):** 5 FPN levels $\times$ 3 sizes $\times$ 3 aspect ratios fitted via $1 - \text{IoU}$ k-means on 1280-scale train boxes (`10..109 px`, mean best IoU `0.648 -> 0.819`), preserving 9 anchors/location.
- **`retina-e2` (Exp B — Focal Loss & Tiny-Object Matching):** Focal $\gamma=1.5, \alpha=0.25$, relaxed anchor Matcher IoU thresholds (`0.5/0.4 -> 0.4/0.3`), and GIoU box regression.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from vdet.paths import P
from vdet.utils import seed_everything
from vdet.models.anchors import fit_kmeans_anchors
seed_everything(42)

In [ ]:
# 1. Inspect EDA-grounded k-means anchors (1280 scale)
print(fit_kmeans_anchors(use_cached_yaml=True))

# 2. Training commands:
# !python scripts/train_tv.py --config configs/retinanet/e0_base.yaml --eval-val
# !python scripts/train_tv.py --config configs/retinanet/e1_kmeans_anchors.yaml --eval-val
# !python scripts/train_tv.py --config configs/retinanet/e2_focal_assign.yaml --eval-val

In [ ]:
# 3. Display F2 vs F1 results table
if P.results_csv.exists():
    df = pd.read_csv(P.results_csv)
    display(df[df['family'].isin(['retinanet', 'faster_rcnn'])][['run_id', 'family', 'exp', 'AP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR500', 'fps', 'gpu_h', 'epochs_done']])